# EduAssist AI — LLMOps Prompt Engineering & Evaluation Assignment

**Goal:** test three prompt versions for an academic question-answering assistant, measure their operational performance (latency, tokens, cost, JSON validity), and promote the best version to production using the MLflow Prompt Registry.

## Model note

All LLM calls in this notebook run **locally** against **`Qwen/Qwen2.5-3B-Instruct`** (ungated, Apache-2.0, ~3B parameters) via Hugging Face `transformers`, on this machine's Apple-Silicon GPU (`mps`). Token counts come from that model's own tokenizer. Cost is estimated using the published hosting rate for `Qwen/Qwen2.5-7B-Instruct` (the smallest Qwen2.5 checkpoint currently listed by hosted providers), sourced from OpenRouter's aggregated pricing API, accessed **2026-11-04**, in **USD** — see `src/eduassist/config.py::PRICING` for the exact numbers and source URL.

The code also implements the real Hugging Face Inference API call path (`src/eduassist/llm_client.py::call_via_hf_inference_api`), used automatically instead of the local model if `HF_TOKEN` is exported (loaded from an environment variable — never hard-coded).

All reusable logic lives in the `src/eduassist/` package; this notebook imports it and focuses on orchestration, evidence capture, and narrative.

## Setup

Imports, path configuration (so `import eduassist` works regardless of the notebook's working directory), and an environment sanity check.

In [2]:
import sys
from pathlib import Path

# Make `import eduassist` work whether Jupyter's cwd is `notebooks/` or the project root.
_CWD = Path.cwd()
PROJECT_ROOT = _CWD if (_CWD / "src").exists() else _CWD.parent
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import os
import platform
import torch
import transformers
import mlflow

print("Project root:         ", PROJECT_ROOT)
print("Python version:        ", platform.python_version())
print("torch version:         ", torch.__version__)
print("transformers version:  ", transformers.__version__)
print("mlflow version:        ", mlflow.__version__)
print("MPS (Apple GPU) available:", torch.backends.mps.is_available())
print("HF_TOKEN set:          ", bool(os.environ.get("HF_TOKEN")))

from eduassist.config import MODEL_ID_LOCAL, PRICING

print("\nModel used for every call:", MODEL_ID_LOCAL)
print("Pricing: ${}/1M input tokens, ${}/1M output tokens ({}, as of {}) - model: {}, source: {}".format(
    PRICING["input_price_per_million"],
    PRICING["output_price_per_million"],
    PRICING["currency"],
    PRICING["pricing_date"],
    PRICING["model_name"],
    PRICING["source_url"],
))

Project root:          /Users/sibasis.mishra/Projects/Personal/MLOPS Assignment-2
Python version:         3.12.11
torch version:          2.15.0
transformers version:   5.22.1
mlflow version:         2.22.5
MPS (Apple GPU) available: True
HF_TOKEN set:           False

Model used for every call: Qwen/Qwen2.5-3B-Instruct
Pricing: $0.1/1M input tokens, $0.2/1M output tokens (USD, as of 2026-11-04) - model: Qwen/Qwen2.5-7B-Instruct, source: https://openrouter.ai/api/v1/models


## Task 1 — Call an LLM and Measure Latency

We call the local model for **five academic questions** (spanning ML, physics, algorithms, biology, and statistics) using **Prompt v1 (Basic)**, recording question ID, prompt version, model name, start time, end time, latency, and response status for every request. The API key / token requirement is satisfied via `HF_TOKEN` (loaded from an environment variable in `src/eduassist/config.py` — never hard-coded); it is simply unset here, which is what triggers the documented local fallback.

The five questions (shared by every task in this notebook) are defined in `src/eduassist/config.py::QUESTIONS`:

In [3]:
import pandas as pd
from eduassist.config import QUESTIONS

pd.DataFrame(QUESTIONS)

,question_id,topic,question
0,Q1,Machine Learning,What is the difference between supervised and ...
1,Q2,Physics,Explain Newton's second law of motion with an ...
2,Q3,Algorithms,What is the time complexity of binary search a...
3,Q4,Biology,Describe the process of photosynthesis in plants.
4,Q5,Statistics,What is the Central Limit Theorem in statistics?


In [4]:
from eduassist.runner import run_prompt_version

# Executable API-call code for Task 1: five real calls to the model, prompt v1 (Basic).
# The first call also lazily loads the model/tokenizer (one-time cost, included in its latency).
task1_records = run_prompt_version("v1")

task1_df = pd.DataFrame(task1_records)
task1_df[
    ["question_id", "prompt_version", "model_name", "start_time", "end_time", "latency_s", "status"]
]

W1105 03:13:57.576000 89806 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

,question_id,prompt_version,model_name,start_time,end_time,latency_s,status
0,Q1,v1,Qwen/Qwen2.5-3B-Instruct,2026-11-05T03:14:10.353,2026-11-05T03:14:41.964,31.6106,success
1,Q2,v1,Qwen/Qwen2.5-3B-Instruct,2026-11-05T03:14:41.968,2026-11-05T03:15:12.608,30.6404,success
2,Q3,v1,Qwen/Qwen2.5-3B-Instruct,2026-11-05T03:15:12.610,2026-11-05T03:15:42.457,29.8473,success
3,Q4,v1,Qwen/Qwen2.5-3B-Instruct,2026-11-05T03:15:42.459,2026-11-05T03:16:12.455,29.9960,success
4,Q5,v1,Qwen/Qwen2.5-3B-Instruct,2026-11-05T03:16:12.458,2026-11-05T03:16:34.045,21.5869,success


In [5]:
latency_summary = task1_df["latency_s"].agg(["mean", "median", "min", "max", "std"]).rename(
    {"mean": "mean_s", "median": "median_s", "min": "min_s", "max": "max_s", "std": "std_s"}
)
latency_summary_df = latency_summary.to_frame(name="value").round(4)
latency_summary_df

,value
mean_s,28.7362
median_s,29.9960
min_s,21.5869
max_s,31.6106
std_s,4.0565


In [6]:
from IPython.display import Markdown, display

slowest = task1_df.loc[task1_df["latency_s"].idxmax()]
fastest = task1_df.loc[task1_df["latency_s"].idxmin()]
mean_latency = latency_summary["mean_s"]

interpretation = f"""**Interpretation — slowest request:** the slowest request was **{slowest['question_id']}**
("*{slowest['question']}*"), taking **{slowest['latency_s']:.2f}s** versus a **{mean_latency:.2f}s** mean and a
**{fastest['latency_s']:.2f}s** fastest request ({fastest['question_id']}). It produced **{slowest['output_tokens']}**
output tokens compared to {fastest['output_tokens']} for the fastest request. Because generation is autoregressive and
runs with greedy decoding on this machine's local GPU (MPS), wall-clock latency is driven almost entirely by **how many
tokens had to be generated**, not network variance (there is no network hop at all in the local path) — a question whose
answer naturally runs longer (more sub-topics to cover, hitting the `max_new_tokens={300}` cap) takes
proportionally longer. A smaller secondary factor is that the very first call in a session pays a one-time
model/tokenizer load and GPU warm-up cost, which is not isolated out here but is shared by whichever question happens to
run first. In a real hosted-API deployment the same pattern holds plus added queuing/network latency, so the
practical mitigation is identical in both environments: cap `max_tokens`/response length and prefer prompts that
elicit concise answers (directly motivating the Task 2 cost-optimization recommendation below)."""

display(Markdown(interpretation))

**Interpretation — slowest request:** the slowest request was **Q1**
("*What is the difference between supervised and unsupervised learning?*"), taking **31.61s** versus a **28.74s** mean and a
**21.59s** fastest request (Q5). It produced **300**
output tokens compared to 213 for the fastest request. Because generation is autoregressive and
runs with greedy decoding on this machine's local GPU (MPS), wall-clock latency is driven almost entirely by **how many
tokens had to be generated**, not network variance (there is no network hop at all in the local path) — a question whose
answer naturally runs longer (more sub-topics to cover, hitting the `max_new_tokens=300` cap) takes
proportionally longer. A smaller secondary factor is that the very first call in a session pays a one-time
model/tokenizer load and GPU warm-up cost, which is not isolated out here but is shared by whichever question happens to
run first. In a real hosted-API deployment the same pattern holds plus added queuing/network latency, so the
practical mitigation is identical in both environments: cap `max_tokens`/response length and prefer prompts that
elicit concise answers (directly motivating the Task 2 cost-optimization recommendation below).

## Task 2 — Count Tokens and Calculate Cost

Token counts for every request come from the actually-executed model's own tokenizer (`Qwen/Qwen2.5-3B-Instruct`), captured in `src/eduassist/llm_client.py` at call time (`input_tokens` = length of the chat-templated prompt; `output_tokens` = number of newly generated tokens). Cost is then computed with the published reference-model rate from `src/eduassist/config.py::PRICING`:

```
Input Cost  = input_tokens  / 1,000,000 × input_price_per_million
Output Cost = output_tokens / 1,000,000 × output_price_per_million
Total Cost  = Input Cost + Output Cost
```

In [7]:
cost_table = task1_df[
    [
        "question_id",
        "input_tokens",
        "output_tokens",
        "total_tokens",
        "input_cost_usd",
        "output_cost_usd",
        "total_cost_usd",
    ]
].copy()
for col in ["input_cost_usd", "output_cost_usd", "total_cost_usd"]:
    cost_table[col] = cost_table[col].round(8)
cost_table

,question_id,input_tokens,output_tokens,total_tokens,input_cost_usd,output_cost_usd,total_cost_usd
0,Q1,50,300,350,0.000005,0.000060,0.000065
1,Q2,49,300,349,0.000005,0.000060,0.000065
2,Q3,49,300,349,0.000005,0.000060,0.000065
3,Q4,47,300,347,0.000005,0.000060,0.000065
4,Q5,48,213,261,0.000005,0.000043,0.000047


In [8]:
from eduassist.costs import projected_monthly_cost
from eduassist.config import MONTHLY_PROJECTED_REQUESTS

DATA_DIR = PROJECT_ROOT / "data"
DATA_DIR.mkdir(exist_ok=True)

# Deliverable: latency_tokens_cost.csv (Task 1 + Task 2 combined, one row per request)
latency_tokens_cost_path = DATA_DIR / "latency_tokens_cost.csv"
task1_df.drop(columns=["response_text"]).to_csv(latency_tokens_cost_path, index=False)
print("Saved:", latency_tokens_cost_path)

avg_cost_per_request = task1_df["total_cost_usd"].mean()
monthly_cost = projected_monthly_cost(avg_cost_per_request, MONTHLY_PROJECTED_REQUESTS)

print(f"\nAverage cost per request (prompt v1, n=5): ${avg_cost_per_request:.8f}")
print(f"Projected monthly cost at {MONTHLY_PROJECTED_REQUESTS:,} requests: ${monthly_cost:,.2f}")

Saved: /Users/sibasis.mishra/Projects/Personal/MLOPS Assignment-2/data/latency_tokens_cost.csv

Average cost per request (prompt v1, n=5): $0.00006138
Projected monthly cost at 10,000 requests: $0.61


In [10]:
capped = int((task1_df["output_tokens"] >= 300).sum())
output_share = task1_df["output_cost_usd"].sum() / task1_df["total_cost_usd"].sum()
price_ratio = PRICING["output_price_per_million"] / PRICING["input_price_per_million"]

recommendation = f"""**Cost-optimization recommendation:** {capped} of 5 requests under Prompt v1 generated the full
`max_new_tokens=300` without naturally stopping — i.e. the model kept elaborating rather than concluding. Output
tokens already account for **{output_share:.0%}** of total cost here even though the output price (${PRICING['output_price_per_million']}/1M)
is only ~{price_ratio:.1f}× the input price (${PRICING['input_price_per_million']}/1M), because answers are so much longer than
questions. The single highest-leverage optimization is therefore **shortening responses** — e.g. adding an explicit
length/format constraint to the prompt (as Prompt v2 does with "clear, concise") or lowering `max_tokens` for
simple factual questions — rather than reducing input size, caching, or switching models, since input tokens are
already a small fraction of the per-request cost here. At scale (10,000 requests/month), trimming average output
length by ~40% would cut the monthly bill by roughly the same proportion of the {output_share:.0%} output-cost share,
without touching accuracy for most questions."""

display(Markdown(recommendation))

**Cost-optimization recommendation:** 4 of 5 requests under Prompt v1 generated the full
`max_new_tokens=300` without naturally stopping — i.e. the model kept elaborating rather than concluding. Output
tokens already account for **92%** of total cost here even though the output price ($0.2/1M)
is only ~2.0× the input price ($0.1/1M), because answers are so much longer than
questions. The single highest-leverage optimization is therefore **shortening responses** — e.g. adding an explicit
length/format constraint to the prompt (as Prompt v2 does with "clear, concise") or lowering `max_tokens` for
simple factual questions — rather than reducing input size, caching, or switching models, since input tokens are
already a small fraction of the per-request cost here. At scale (10,000 requests/month), trimming average output
length by ~40% would cut the monthly bill by roughly the same proportion of the 92% output-cost share,
without touching accuracy for most questions.

## Task 3 — Prompt Versioning and MLflow Evaluation

We now register all three prompt versions in the **MLflow Prompt Registry** under one registered prompt name
(`eduassist_answer_prompt`), run each version against the same five questions, log per-version metrics to the
MLflow experiment `eduassist_llmops_assignment`, compare the three, and promote the winner with the
**`production`** alias.

Tracking store: local `mlruns/` folder at the project root (`file:` backend), so everything below is fully
reproducible offline and viewable with `mlflow ui` (see the README for the exact command).

In [11]:
from mlflow import MlflowClient
from eduassist.config import MLFLOW_EXPERIMENT_NAME, PROMPT_TEMPLATES, PRODUCTION_ALIAS

MLRUNS_DIR = PROJECT_ROOT / "mlruns"
mlflow.set_tracking_uri(f"file:{MLRUNS_DIR}")
mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)
client = MlflowClient()

print("MLflow tracking URI:", mlflow.get_tracking_uri())
print("Experiment:", MLFLOW_EXPERIMENT_NAME)

REGISTERED_PROMPT_NAME = "eduassist_answer_prompt"

2026/11/05 03:18:17 INFO mlflow.tracking.fluent: Experiment with name 'eduassist_llmops_assignment' does not exist. Creating a new experiment.


MLflow tracking URI: file:/Users/sibasis.mishra/Projects/Personal/MLOPS Assignment-2/mlruns
Experiment: eduassist_llmops_assignment


In [12]:
# Register v1, v2, v3 with meaningful commit messages. Re-running this cell is safe: MLflow
# creates a new version only when the template text actually changed.
registered_versions = {}
for version_key, spec in PROMPT_TEMPLATES.items():
    prompt = mlflow.register_prompt(
        name=REGISTERED_PROMPT_NAME,
        template=spec["template"],
        commit_message=spec["commit_message"],
        tags={"version_label": version_key, "style": spec["label"]},
    )
    registered_versions[version_key] = prompt
    print(f"Registered '{REGISTERED_PROMPT_NAME}' {version_key} ({spec['label']}) -> registry version {prompt.version}")

Registered 'eduassist_answer_prompt' v1 (Basic) -> registry version 1
Registered 'eduassist_answer_prompt' v2 (Structured) -> registry version 2
Registered 'eduassist_answer_prompt' v3 (JSON Output) -> registry version 3


In [14]:
# Run each prompt version on the same five questions (15 calls total: 3 versions x 5 questions).
task3_records = []
for version_key in ["v1", "v2", "v3"]:
    print(f"Running prompt {version_key} ({PROMPT_TEMPLATES[version_key]['label']}) on 5 questions...")
    task3_records.extend(run_prompt_version(version_key))

task3_df = pd.DataFrame(task3_records)

# Deliverable: prompt_evaluation_results.csv
prompt_evaluation_results_path = DATA_DIR / "prompt_evaluation_results.csv"
task3_df.to_csv(prompt_evaluation_results_path, index=False)
print("\nSaved:", prompt_evaluation_results_path, f"({len(task3_df)} rows)")

task3_df[
    ["question_id", "prompt_version", "status", "latency_s", "input_tokens", "output_tokens", "total_cost_usd", "valid_json"]
]

Running prompt v1 (Basic) on 5 questions...
Running prompt v2 (Structured) on 5 questions...
Running prompt v3 (JSON Output) on 5 questions...

Saved: /Users/sibasis.mishra/Projects/Personal/MLOPS Assignment-2/data/prompt_evaluation_results.csv (15 rows)


,question_id,prompt_version,status,latency_s,input_tokens,output_tokens,total_cost_usd,valid_json
0,Q1,v1,success,32.4452,50,300,0.000065,False
1,Q2,v1,success,31.3554,49,300,0.000065,False
2,Q3,v1,success,30.8035,49,300,0.000065,False
3,Q4,v1,success,30.1483,47,300,0.000065,False
4,Q5,v1,success,21.7635,48,213,0.000047,False
5,Q1,v2,success,29.4934,72,292,0.000066,False
6,Q2,v2,success,30.3898,71,300,0.000067,False
7,Q3,v2,success,29.7801,71,296,0.000066,False
8,Q4,v2,success,28.1883,69,268,0.000061,False
9,Q5,v2,success,26.7298,70,261,0.000059,False


In [15]:
comparison_rows = []
run_ids = {}

for version_key in ["v1", "v2", "v3"]:
    subset = task3_df[task3_df["prompt_version"] == version_key]
    spec = PROMPT_TEMPLATES[version_key]
    prompt_version_obj = registered_versions[version_key]

    metrics = {
        "avg_latency_seconds": float(subset["latency_s"].mean()),
        "avg_input_tokens": float(subset["input_tokens"].mean()),
        "avg_output_tokens": float(subset["output_tokens"].mean()),
        "avg_cost_per_request": float(subset["total_cost_usd"].mean()),
        # MLflow aggregates boolean feedback into an average, which is exactly the
        # Valid JSON Responses / Total Responses pass rate the assignment asks for.
        "valid_json": float(subset["valid_json"].mean()),
    }

    run_name = f"prompt_{version_key}_{spec['label'].lower().replace(' ', '_')}"
    with mlflow.start_run(run_name=run_name) as run:
        mlflow.log_param("prompt_name", REGISTERED_PROMPT_NAME)
        mlflow.log_param("prompt_registry_version", prompt_version_obj.version)
        mlflow.log_param("prompt_label", spec["label"])
        mlflow.log_param("commit_message", spec["commit_message"])
        mlflow.log_param("model_name", subset["model_name"].iloc[0])
        mlflow.log_param("n_questions", len(subset))
        mlflow.log_metrics(metrics)
        # Links this run to the exact registered prompt version it evaluated -
        # this is what lets the MLflow UI show prompt <-> run lineage / diff evidence.
        client.log_prompt(run.info.run_id, prompt_version_obj)

        version_csv = DATA_DIR / f"prompt_evaluation_{version_key}.csv"
        subset.to_csv(version_csv, index=False)
        mlflow.log_artifact(str(version_csv), artifact_path="per_request_results")

        run_ids[version_key] = run.info.run_id

    comparison_rows.append(
        {
            "Prompt Version": f"{version_key} ({spec['label']})",
            "Avg Latency (s)": round(metrics["avg_latency_seconds"], 3),
            "Avg Input Tokens": round(metrics["avg_input_tokens"], 1),
            "Avg Output Tokens": round(metrics["avg_output_tokens"], 1),
            "Avg Cost ($)": round(metrics["avg_cost_per_request"], 8),
            "Valid JSON Rate": metrics["valid_json"],
        }
    )
    print(f"Logged MLflow run '{run_name}' (run_id={run_ids[version_key]}) for prompt version {version_key}")

comparison_df = pd.DataFrame(comparison_rows)
comparison_df

Logged MLflow run 'prompt_v1_basic' (run_id=3ed7cbb62461481095d1a4c42660a39d) for prompt version v1
Logged MLflow run 'prompt_v2_structured' (run_id=8b61acd1edfc451389696ba11838a871) for prompt version v2
Logged MLflow run 'prompt_v3_json_output' (run_id=4ff6f135afae4349be29e8177b5a4c75) for prompt version v3


,Prompt Version,Avg Latency (s),Avg Input Tokens,Avg Output Tokens,Avg Cost ($),Valid JSON Rate
0,v1 (Basic),29.303,48.6,282.6,0.000061,0.0
1,v2 (Structured),28.916,70.6,283.4,0.000064,0.0
2,v3 (JSON Output),15.923,104.6,155.4,0.000042,1.0


### Selecting and promoting the best version

The product requirement is a **strict JSON contract** (`answer` / `confidence` / `key_points`). A version only
qualifies for production if it actually satisfies that contract — latency and cost are tie-breakers *among*
JSON-compliant versions, not substitutes for compliance. Concretely: among versions with the highest `Valid JSON
Rate`, pick the one with the lowest `Avg Cost ($)`.

In [16]:
max_valid_json_rate = comparison_df["Valid JSON Rate"].max()
json_capable = comparison_df[comparison_df["Valid JSON Rate"] == max_valid_json_rate]
best_row = json_capable.sort_values("Avg Cost ($)").iloc[0]
best_version_key = best_row["Prompt Version"].split(" ")[0]

comparison_df["Recommended?"] = comparison_df["Prompt Version"].apply(
    lambda v: "Yes" if v.split(" ")[0] == best_version_key else "No"
)

# Promote the winner to production using the MLflow Prompt Registry alias.
best_prompt_version = registered_versions[best_version_key]
client.set_prompt_alias(REGISTERED_PROMPT_NAME, PRODUCTION_ALIAS, best_prompt_version.version)

promoted = mlflow.load_prompt(f"prompts:/{REGISTERED_PROMPT_NAME}@{PRODUCTION_ALIAS}")
print(
    f"Promoted prompt version {best_version_key} (registry v{best_prompt_version.version}) to alias "
    f"'{PRODUCTION_ALIAS}'. Confirmed alias -> version {promoted.version}, aliases={promoted.aliases}"
)

comparison_df

Promoted prompt version v3 (registry v3) to alias 'production'. Confirmed alias -> version 3, aliases=['production']


,Prompt Version,Avg Latency (s),Avg Input Tokens,Avg Output Tokens,Avg Cost ($),Valid JSON Rate,Recommended?
0,v1 (Basic),29.303,48.6,282.6,0.000061,0.0,No
1,v2 (Structured),28.916,70.6,283.4,0.000064,0.0,No
2,v3 (JSON Output),15.923,104.6,155.4,0.000042,1.0,Yes


In [17]:
row = {r["Prompt Version"].split(" ")[0]: r for r in comparison_rows}
v1, v2, v3 = row["v1"], row["v2"], row["v3"]
best = row[best_version_key]

# Describe cost/latency truthfully relative to whatever was actually measured, instead of
# asserting a specific outcome - the only fixed claim is the JSON-compliance gate itself.
other_keys = [k for k in ["v1", "v2", "v3"] if k != best_version_key]
cheaper_than_all = all(best["Avg Cost ($)"] <= row[k]["Avg Cost ($)"] for k in other_keys)
faster_than_all = all(best["Avg Latency (s)"] <= row[k]["Avg Latency (s)"] for k in other_keys)
if cheaper_than_all and faster_than_all:
    tradeoff_sentence = (
        f"On this question set, {best_version_key} is not a quality/cost trade-off at all: it is "
        f"simultaneously the only contract-compliant version **and** the cheapest and fastest of the three."
    )
elif cheaper_than_all or faster_than_all:
    cheaper_txt = "cheapest" if cheaper_than_all else "not the cheapest"
    faster_txt = "fastest" if faster_than_all else "not the fastest"
    tradeoff_sentence = (
        f"On this question set, {best_version_key} is also the {cheaper_txt} and {faster_txt} of the three, "
        f"so the only real trade-off being accepted is on whichever of latency/cost it does not lead."
    )
else:
    tradeoff_sentence = (
        f"On this question set, {best_version_key} is **not** the cheapest or fastest version - that cost/latency "
        f"premium is accepted deliberately, because contract compliance (valid JSON) is a hard product "
        f"requirement that v1/v2 do not meet at all, not a metric to be traded off against speed or cost."
    )

justification = f"""### Promotion justification

Prompt **{best_version_key} ({PROMPT_TEMPLATES[best_version_key]['label']})** is promoted to the **`production`**
alias. **Quality / contract compliance** is the deciding factor: EduAssist's product requirement is a strict JSON
response (`answer`, `confidence`, `key_points`), and only v3 was designed to satisfy it — its measured
**valid-JSON rate is {v3['Valid JSON Rate']:.0%}** across the five evaluation questions, parsed with a strict
schema check (exact keys, `confidence` constrained to high/medium/low, `key_points` as a list), versus v1
and v2, which were never asked to produce JSON and scored **{v1['Valid JSON Rate']:.0%}** and
**{v2['Valid JSON Rate']:.0%}** respectively on that same check. A downstream application parsing these
responses programmatically cannot safely consume v1/v2 output at all, regardless of how fluent the prose is, so
JSON validity functions as a hard gate rather than one metric among equals.

On **latency**, v3 averaged **{v3['Avg Latency (s)']:.2f}s** per request versus **{v1['Avg Latency (s)']:.2f}s**
(v1) and **{v2['Avg Latency (s)']:.2f}s** (v2) — the JSON constraint tends to make the model stop as soon as the
closing brace is emitted, rather than continuing to elaborate, which is reflected in **tokens**: v3 used
**{v3['Avg Output Tokens']:.0f}** average output tokens versus **{v1['Avg Output Tokens']:.0f}** (v1) and
**{v2['Avg Output Tokens']:.0f}** (v2), at a small input-token premium from the longer instruction
({v3['Avg Input Tokens']:.0f} vs {v1['Avg Input Tokens']:.0f}/{v2['Avg Input Tokens']:.0f} tokens) for the schema
description. Combined, this gives v3 an **average cost per request of ${v3['Avg Cost ($)']:.8f}**, compared with
${v1['Avg Cost ($)']:.8f} (v1) and ${v2['Avg Cost ($)']:.8f} (v2). {tradeoff_sentence} Given all five dimensions
required by the assignment - quality (schema compliance), JSON validity, latency, tokens, and cost - **{best_version_key}**
is the clear, defensible choice for the `production` alias."""

display(Markdown(justification))
print(f"\n(word count: {len(justification.split())})")

### Promotion justification

Prompt **v3 (JSON Output)** is promoted to the **`production`**
alias. **Quality / contract compliance** is the deciding factor: EduAssist's product requirement is a strict JSON
response (`answer`, `confidence`, `key_points`), and only v3 was designed to satisfy it — its measured
**valid-JSON rate is 100%** across the five evaluation questions, parsed with a strict
schema check (exact keys, `confidence` constrained to high/medium/low, `key_points` as a list), versus v1
and v2, which were never asked to produce JSON and scored **0%** and
**0%** respectively on that same check. A downstream application parsing these
responses programmatically cannot safely consume v1/v2 output at all, regardless of how fluent the prose is, so
JSON validity functions as a hard gate rather than one metric among equals.

On **latency**, v3 averaged **15.92s** per request versus **29.30s**
(v1) and **28.92s** (v2) — the JSON constraint tends to make the model stop as soon as the
closing brace is emitted, rather than continuing to elaborate, which is reflected in **tokens**: v3 used
**155** average output tokens versus **283** (v1) and
**283** (v2), at a small input-token premium from the longer instruction
(105 vs 49/71 tokens) for the schema
description. Combined, this gives v3 an **average cost per request of $0.00004154**, compared with
$0.00006138 (v1) and $0.00006374 (v2). On this question set, v3 is not a quality/cost trade-off at all: it is simultaneously the only contract-compliant version **and** the cheapest and fastest of the three. Given all five dimensions
required by the assignment - quality (schema compliance), JSON validity, latency, tokens, and cost - **v3**
is the clear, defensible choice for the `production` alias.


(word count: 268)


## Viewing the MLflow evidence (experiment, Prompt Registry, diff, production alias)

With this notebook's cells executed, the local `mlruns/` store at the project root now contains the experiment,
the three logged runs (one per prompt version), and the registered prompt with its three versions and the
`production` alias. To view it:

```bash
cd "MLOPS Assignment-2"
mlflow ui --backend-store-uri file:./mlruns --port 5000
```

Then open `http://127.0.0.1:5000` and capture (see `screenshots/` for the saved evidence referenced in the
report):

1. **Experiments → `eduassist_llmops_assignment`** — the three runs with their logged metrics.
2. **Prompts (top-level nav) → `eduassist_answer_prompt`** — all three versions (v1/v2/v3).
3. Open two versions and use **Compare** — the prompt diff view.
4. The prompt's version detail page showing the **`production`** alias attached to the winning version.

## Summary & deliverables checklist

- [x] Task 1: executable API-call code, latency table for 5 requests, summary statistics, interpretation paragraph.
- [x] Task 2: token/cost table, `latency_tokens_cost.csv`, monthly cost projection, cost-optimization recommendation.
- [x] Task 3: prompts registered with commit messages, 15 calls logged to MLflow, per-version metrics logged,
      `prompt_evaluation_results.csv`, comparison table, `production` alias promotion with justification.
- [ ] MLflow screenshots (experiment, Prompt Registry, metrics, diff, production alias) — captured separately into
      `screenshots/` per the instructions above, and embedded in the final PDF report.
- [ ] Final PDF report, README, and `requirements.txt` — see project root.